# Astra Day 4: Naive RAG Live Walkthrough (Google Colab Edition)
### 8-Stage Retrieval-Augmented Generation via NVIDIA NIM & Pinecone

Welcome to the hands-on lab for **Day 4: RAG Fundamentals**!

In this notebook, we trace the complete **8-stage Naive RAG pipeline** against the curriculum diagram (Slide 13):

```
========================= OFFLINE INDEXING PIPELINE =========================
[Raw Source File] 
       │
       ▼ (Stage 1: Document Ingestion)
[Full Text String] 
       │
       ▼ (Stage 2: Chunking with Overlap)
[Discrete Passages / Chunks] 
       │
       ▼ (Stage 3: Vector Embedding via NVIDIA NIM)
[2048-Dimensional Vectors] 
       │
       ▼ (Stage 4: Storage)
[(Pinecone Cloud / Local Vector DB)]

========================== ONLINE QUERY PIPELINE ============================
[User Live Query]
       │
       ▼ (Stage 5: Query Embedding via NVIDIA NIM)
[Query Vector (2048-dim)]
       │
       ▼ (Stage 6: Vector Retrieval / Cosine Similarity)
[Top-K Most Similar Chunks]
       │
       ▼ (Stage 7: Context Injection)
[Augmented System Prompt]
       │
       ▼ (Stage 8: Grounded LLM Generation via NVIDIA NIM)
[Factually Grounded Response]
```

---
### Dual-Mode Vector Database Strategy
* **Mode A (Zero Setup / Instant):** If you don't have a Pinecone account yet, the notebook automatically uses an internal mathematical **Cosine Similarity** vector store. You can run all 8 stages immediately without signing up for anything!
* **Mode B (Pinecone Cloud):** When you paste your free Pinecone API key, the notebook automatically creates a **Pinecone Serverless Index** on AWS and performs cloud vector searches.

## Step 0: Install Dependencies (Google Colab)
Run the cell below to install the required libraries in your Colab runtime.
*(Note: If running locally in VS Code, ensure your `.venv` is selected).*

In [ ]:
# Install required packages quietly in Google Colab
!pip install -q openai python-dotenv pinecone

print("[OK] Dependencies installed successfully!")

## Step 1: Environment & API Key Setup
We load credentials using a student-friendly 3-tier fallback:
1. **Google Colab Secrets:** The 🔑 key icon on the left sidebar.
2. **Local `.env` / `.env.local`:** If you cloned the repository.
3. **Interactive Prompt:** If no key is found, Colab will securely prompt you to paste it.

> [!TIP]
> **Get your free NVIDIA NIM API key:** [https://build.nvidia.com/](https://build.nvidia.com/)  
> **Get your free Pinecone API key (Optional):** [https://app.pinecone.io/](https://app.pinecone.io/)

In [ ]:
import os
import math
import json
import getpass
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# 1. Attempt to fetch from Colab Secrets (Left sidebar 🔑 icon)
nvidia_key = ""
pinecone_key = ""

try:
    from google.colab import userdata
    try:
        nvidia_key = userdata.get("NVIDIA_API_KEY")
    except Exception:
        pass
    try:
        pinecone_key = userdata.get("PINECONE_API_KEY")
    except Exception:
        pass
except ImportError:
    pass

# 2. Fallback to local .env / .env.local if present
load_dotenv(dotenv_path=".env.local", override=True)
load_dotenv()

if not nvidia_key:
    nvidia_key = os.getenv("NVIDIA_API_KEY", "").strip()

if not pinecone_key:
    pinecone_key = os.getenv("PINECONE_API_KEY", "").strip()

# 3. Interactive prompt if still missing
if not nvidia_key:
    print("🔑 Please enter your NVIDIA NIM API Key (starts with nvapi-):")
    nvidia_key = getpass.getpass("NVIDIA API Key: ").strip()

os.environ["NVIDIA_API_KEY"] = nvidia_key

# Model & Hyperparameter Settings
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
CHAT_MODEL = "openai/gpt-oss-20b"
EMBED_MODEL = "nvidia/nemotron-3-embed-1b"
PINECONE_INDEX_NAME = "naive-rag-colab"

CHUNK_SIZE = 500       # Size of each chunk in characters
CHUNK_OVERLAP = 100    # Boundary overlap to keep sentences intact
TOP_K = 3              # Number of nearest passages to retrieve

# Initialize OpenAI client pointing to NVIDIA NIM
client = OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=nvidia_key,
)

print("\n[OK] Configuration complete!")
print(f"[*] NVIDIA NIM Connected : Yes (Model: {CHAT_MODEL})")
print(f"[*] Embedding Model      : {EMBED_MODEL} (2048-dim)")
if pinecone_key:
    print(f"[*] Vector DB Mode       : PINECONE CLOUD (Index: {PINECONE_INDEX_NAME})")
else:
    print("[*] Vector DB Mode       : LOCAL COSINE FALLBACK (Zero setup required!)")

## Step 2: Auto-Provision Knowledge Base
**The Goal:** Guarantee the document exists in Colab so you never encounter `FileNotFoundError`.  
**Dataset:** AstraTech Solutions Employee Handbook (internal rules for remote stipends, Wi-Fi passwords, and PTO).

In [ ]:
# Create data directory in Colab runtime
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
handbook_file = data_dir / "astratech_employee_handbook.txt"

# If the file does not exist (e.g. running in fresh Colab session), write it automatically!
if not handbook_file.exists():
    handbook_content = """# ASTRATECH SOLUTIONS: EMPLOYEE HANDBOOK & COMPANY POLICIES
# DOCUMENT ID: AT-HR-2026-V2
# AUDIENCE: ALL NEW HIRES & ENGINEERING STAFF

---

## SECTION 1: WORKING HOURS & HYBRID WORK POLICY
AstraTech Solutions operates on a flexible hybrid schedule designed to balance in-person collaboration with focused deep work. Core business collaboration hours are 10:00 AM to 4:00 PM local time, during which team members are expected to be reachable on Slack. All full-time employees work from the main office three days a week (Mondays, Tuesdays, and Thursdays). Remote work from home is permitted on Wednesdays and Fridays. To protect uninterrupted coding time, Tuesday afternoons from 1:00 PM to 5:00 PM are designated as company-wide "No-Meeting Focus Blocks."

## SECTION 2: PAID TIME OFF (PTO) & WELLNESS BENEFITS
All full-time employees receive 24 days of paid time off (PTO) per calendar year, accrued monthly. Employees may carry over a maximum of 5 unused PTO days into the following calendar year. Sick leave is fully paid and allows up to 5 consecutive business days without requiring a doctor's medical note. In addition to standard health insurance, AstraTech provides an annual Wellness Stipend of $800 per employee, reimbursable for gym memberships, yoga classes, or mental health counseling apps.

## SECTION 3: HARDWARE STIPEND & HOME OFFICE SETUP
Every new engineering hire chooses between a 16-inch MacBook Pro (M3 Max, 36GB Unified Memory) or a Lenovo ThinkPad P16 mobile workstation upon joining. To support comfortable remote work on Wednesdays and Fridays, AstraTech provides a one-time remote home office setup stipend of $1,200. This stipend covers ergonomic chairs, standing desks, 4K external monitors, or noise-cancelling headphones. All home office expense receipts must be submitted via Expensify within 30 days of purchase.

## SECTION 4: OFFICE ACCESS, VISITOR WI-FI & CAFETERIA
The AstraTech campus is accessible 24 hours a day, 7 days a week using your issued RFID employee security badge. If visitors or guests require internet access, the office guest Wi-Fi network SSID is AstraTech-Guest and the password is CoffeeAndCode2026!. Employees commuting by bicycle can access the secure indoor bike storage room located on Basement Level B1; the door keypad unlock code is 7492#. Complimentary catered hot lunch is served daily Monday through Thursday at 12:30 PM on the 4th-floor cafeteria terrace.

## SECTION 5: ANNUAL LEARNING & CONFERENCE STIPEND
To foster continuous engineering growth, each team member is allocated an annual Professional Learning Budget of $2,500 per fiscal year. This budget can be used for O'Reilly technical books, online course subscriptions, and industry certification exams (including NVIDIA NIM, AWS, and GCP certifications). For attending domestic or international developer conferences, travel expense requests must be submitted to your direct engineering manager at least 45 days prior to the conference date.
"""
    with open(handbook_file, "w", encoding="utf-8") as f:
        f.write(handbook_content.strip())
    print(f"[OK] Created employee handbook in Colab: {handbook_file}")
else:
    print(f"[OK] Handbook already exists: {handbook_file}")

## Stage 1: Document Ingestion (`load_document`)
**The Goal:** Read the source text file into Python memory.  
**The "Why":** We verify document integrity and measure raw character and word counts.

In [ ]:
def load_document(file_path: str) -> str:
    path_obj = Path(file_path)
    if not path_obj.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    with open(path_obj, "r", encoding="utf-8") as f:
        text = f.read()
    return text.strip()

raw_document = load_document(str(handbook_file))

print(f"[STAGE 1 COMPLETE] Ingested document: {handbook_file.name}")
print(f"[*] Character Count : {len(raw_document)} characters")
print(f"[*] Word Count      : {len(raw_document.split())} words")
print("\n--- Document Excerpt (First 200 chars) ---")
print(raw_document[:200] + "...")

## Stage 2: Chunking with Overlap (`chunk_document`)
**The Goal:** Break the document into 500-character passages with a 100-character sliding-window overlap.  
**The "Why":** Overlapping prevents words or crucial sentences (like passwords or stipend dollar amounts) from being truncated across chunk boundaries.

In [ ]:
def chunk_document(
    text: str,
    chunk_size: int = 500,
    chunk_overlap: int = 100,
) -> list[dict]:
    step = chunk_size - chunk_overlap
    chunks = []
    start = 0
    chunk_idx = 0

    while start < len(text):
        end = start + chunk_size
        chunk_slice = text[start:end]
        trimmed = chunk_slice.strip()

        if len(trimmed) > 0:
            record = {
                "id": f"chunk_{chunk_idx}",
                "text": trimmed,
                "start": start,
                "end": start + len(chunk_slice),
            }
            chunks.append(record)
            chunk_idx = chunk_idx + 1

        start = start + step

    return chunks

chunks = chunk_document(raw_document, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f"[STAGE 2 COMPLETE] Created {len(chunks)} chunks (Size: {CHUNK_SIZE} chars, Overlap: {CHUNK_OVERLAP} chars)\n")

# Inspect the chunks created
for c in chunks[:4]:
    clean_preview = c["text"].replace("\n", " ")
    if len(clean_preview) > 85:
        clean_preview = clean_preview[:85] + "..."
    print(f"  [{c['id']}] ({len(c['text'])} chars): "{clean_preview}"")

## Stage 3: Vector Embedding via NVIDIA NIM (`embed_chunks`)
**The Goal:** Convert each text chunk into a dense 2048-dimensional vector representation.  
**The "Why":** The embedding model (`nvidia/nemotron-3-embed-1b`) maps semantic concepts into geometric space. Passages with related meanings point in similar directions.

In [ ]:
def embed_chunks(chunk_list: list[dict], model_name: str) -> list[dict]:
    texts_to_embed = []
    for c in chunk_list:
        texts_to_embed.append(c["text"])

    print(f"[*] Calling NVIDIA NIM Embeddings API ({model_name})...")
    response = client.embeddings.create(
        input=texts_to_embed,
        model=model_name,
    )

    enriched = []
    idx = 0
    for item in response.data:
        record = {
            "id": chunk_list[idx]["id"],
            "text": chunk_list[idx]["text"],
            "embedding": item.embedding,
        }
        enriched.append(record)
        idx = idx + 1

    return enriched

embedded_chunks = embed_chunks(chunks, model_name=EMBED_MODEL)
vector_dim = len(embedded_chunks[0]["embedding"])

print(f"\n[STAGE 3 COMPLETE] Successfully generated {len(embedded_chunks)} embedding vectors!")
print(f"[*] Vector Dimensions : {vector_dim} coordinates per chunk")
print(f"[*] Sample values for [{embedded_chunks[0]['id']}]: {[round(x, 4) for x in embedded_chunks[0]['embedding'][:5]]} ...")

## Stage 4: Storage in Vector Database (`store_in_vector_db`)
**The Goal:** Persist chunk vectors and metadata into Pinecone or local storage.  
**The "Why":** If `PINECONE_API_KEY` is present, we initialize a serverless index on AWS. If omitted, we maintain the vectors in Colab memory using pure cosine math.

In [ ]:
pinecone_index = None
local_vector_db = []  # Pre-initialized so retrieve_top_k() can always reference it

if pinecone_key:
    from pinecone import Pinecone, ServerlessSpec
    print(f"[*] Connecting to Pinecone Cloud...")
    pc = Pinecone(api_key=pinecone_key)

    # Check if index exists, create if not
    existing_indexes = [idx.name for idx in pc.list_indexes()]
    if PINECONE_INDEX_NAME not in existing_indexes:
        print(f"[*] Creating Pinecone Serverless Index: '{PINECONE_INDEX_NAME}' (dim={vector_dim})...")
        pc.create_index(
            name=PINECONE_INDEX_NAME,
            dimension=vector_dim,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
        print(f"[OK] Pinecone index created successfully.")
    
    pinecone_index = pc.Index(PINECONE_INDEX_NAME)

    # Format vectors for Pinecone upsert
    vectors_to_upsert = []
    for item in embedded_chunks:
        vectors_to_upsert.append({
            "id": item["id"],
            "values": item["embedding"],
            "metadata": {"text": item["text"]},
        })

    pinecone_index.upsert(vectors=vectors_to_upsert)
    print(f"[STAGE 4 COMPLETE] Upserted {len(vectors_to_upsert)} vectors to Pinecone Index '{PINECONE_INDEX_NAME}'.")

else:
    print("[*] Pinecone key not provided -> Storing in Colab Local Vector Database.")
    # Local fallback storage in memory
    local_vector_db = []
    for item in embedded_chunks:
        local_vector_db.append({
            "id": item["id"],
            "embedding": item["embedding"],
            "text": item["text"],
        })
    print(f"[STAGE 4 COMPLETE] Stored {len(local_vector_db)} vectors in Local Vector Database.")

print("\n" + "=" * 65)
print("[INDEXING PIPELINE COMPLETE -- KNOWLEDGE BASE IS READY FOR QUERIES!]")
print("=" * 65)

---
## The Teaching Hook: Asking the Raw LLM WITHOUT RAG
**The Goal:** Ask the foundation model our test question directly with NO context.  
**The "Why":** Demonstrates the value of RAG. The base LLM does not know AstraTech's private policies and will give a generic guess.

In [ ]:
test_question = (
    "How much is the one-time remote home office setup stipend, "
    "and what is the Wi-Fi password for office visitors?"
)

print(f"QUESTION ASKED: \"{test_question}\"\n")
print("--- RAW LLM RESPONSE (WITHOUT RAG) ---")

raw_response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant. Answer the user's question directly."},
        {"role": "user", "content": test_question},
    ],
    temperature=0.2,
    max_tokens=600,  # gpt-oss-20b is a reasoning model — needs room for <think> tokens
)

print(raw_response.choices[0].message.content)
print("---------------------------------------")
print("\n[TEACHING POINT]: Notice how the base LLM rambles generically or guesses '$100-$500'.")
print("Now let's see how RAG retrieves the exact policy in Stages 5 to 8!")

## Stages 5 & 6: Query Embedding & Vector Retrieval
**The Goal:** Embed the live question into 2048 dimensions, then query Pinecone or use Cosine Similarity to find the Top-3 nearest chunks.

In [ ]:
def calculate_cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    dot_product = 0.0
    norm_a_sq = 0.0
    norm_b_sq = 0.0

    idx = 0
    while idx < len(vec_a):
        a_val = vec_a[idx]
        b_val = vec_b[idx]
        dot_product = dot_product + (a_val * b_val)
        norm_a_sq = norm_a_sq + (a_val * a_val)
        norm_b_sq = norm_b_sq + (b_val * b_val)
        idx = idx + 1

    if norm_a_sq <= 0.0 or norm_b_sq <= 0.0:
        return 0.0

    return dot_product / (math.sqrt(norm_a_sq) * math.sqrt(norm_b_sq))


def embed_query(query_text: str, model_name: str) -> list[float]:
    response = client.embeddings.create(
        input=[query_text],
        model=model_name,
    )
    return response.data[0].embedding


def retrieve_top_k(query_vec: list[float], k: int = 3) -> list[dict]:
    # Branch A: Retrieve from Pinecone Cloud if active
    if pinecone_index is not None:
        query_res = pinecone_index.query(
            vector=query_vec,
            top_k=k,
            include_metadata=True,
        )
        results = []
        for match in query_res.matches:
            results.append({
                "id": match.id,
                "score": float(match.score),
                "text": match.metadata["text"],
            })
        return results

    # Branch B: Retrieve from Local Vector Database (Cosine Similarity)
    scored = []
    for item in local_vector_db:
        sim = calculate_cosine_similarity(query_vec, item["embedding"])
        scored.append({
            "id": item["id"],
            "score": sim,
            "text": item["text"],
        })

    def get_score(entry: dict) -> float:
        return entry["score"]

    scored.sort(key=get_score, reverse=True)

    top_matches = []
    count = 0
    for entry in scored:
        if count < k:
            top_matches.append(entry)
            count = count + 1

    return top_matches


# 1. Stage 5: Query Embedding
query_vector = embed_query(test_question, model_name=EMBED_MODEL)
print(f"[STAGE 5 COMPLETE] Query embedded into {len(query_vector)} dimensions.\n")

# 2. Stage 6: Retrieval
retrieved_chunks = retrieve_top_k(query_vector, k=TOP_K)
print(f"[STAGE 6 COMPLETE] Retrieved Top-{len(retrieved_chunks)} relevant passages:\n")

for rank, match in enumerate(retrieved_chunks, 1):
    preview = match["text"].replace("\n", " ")
    if len(preview) > 105:
        preview = preview[:105] + "..."
    print(f"Match #{rank} [{match['id']}] | Cosine Similarity: {match['score']:.4f} ({match['score']*100:.1f}%)")
    print(f"  Excerpt: \"{preview}\"\n")

## Stages 7 & 8: Context Injection & Grounded Generation
**The Goal:** Inject the retrieved chunks into a prompt with strict grounding instructions, then generate the answer with NVIDIA NIM.

In [ ]:
# Stage 7: Context Injection
context_blocks = []
for idx, chunk in enumerate(retrieved_chunks, 1):
    header = f"--- REFERENCE DOCUMENT {idx} ({chunk['id']}, Similarity: {chunk['score']:.4f}) ---"
    context_blocks.append(f"{header}\n{chunk['text']}")

combined_context = "\n\n".join(context_blocks)

augmented_prompt = f"""You are the official HR & Workplace AI assistant for AstraTech Solutions.
Answer the user's inquiry accurately, professionally, and EXCLUSIVELY based on the verified reference context below.
If the answer cannot be determined strictly from the context, respond: "I do not have sufficient information in the AstraTech employee handbook to answer that question."
Do not speculate, assume, or hallucinate outside the given text.

=== BEGIN VERIFIED REFERENCE CONTEXT ===
{combined_context}
=== END VERIFIED REFERENCE CONTEXT ===

USER QUESTION:
{test_question}

GROUNDED ANSWER:"""

print(f"[STAGE 7 COMPLETE] Assembled prompt with {len(combined_context)} chars of context.\n")

# Stage 8: Grounded Generation
print(f"[*] Calling NVIDIA NIM Chat Model ({CHAT_MODEL})...")
rag_response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "You are a precise technical AI assistant grounded strictly in company policies."},
        {"role": "user", "content": augmented_prompt},
    ],
    temperature=0.1,  # Low temperature for factual precision
    max_tokens=1024,  # Reasoning model: needs extra tokens for internal <think> step
)

final_answer = rag_response.choices[0].message.content.strip()

print("\n" + "=" * 65)
print("[STAGE 8 COMPLETE] FINAL NAIVE RAG GROUNDED ANSWER:")
print("=" * 65)
print(final_answer)
print("=" * 65)

---
## Student Sandbox: Test Your Own Questions!
Try asking any question from the employee handbook:
* *"How many days of unused PTO can an employee roll over into the next year?"*
* *"What is the annual professional learning and conference budget?"*
* *"What is the entrance code for the basement bicycle storage room?"*

In [ ]:
custom_question = "What is the entrance code for the basement bicycle storage room?"

# 1. Embed query
q_vec = embed_query(custom_question, model_name=EMBED_MODEL)

# 2. Retrieve top chunks
matches = retrieve_top_k(q_vec, k=TOP_K)

# 3. Context Injection
ctx = "\n\n".join([f"[{m['id']}]: {m['text']}" for m in matches])
prompt = f"""Answer EXCLUSIVELY based on the context below:
{ctx}

Question: {custom_question}
Answer:"""

# 4. Generate answer
res = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.1,
    max_tokens=600,  # Extra tokens for reasoning model
)

print(f"QUESTION : {custom_question}")
print(f"ANSWER   : {res.choices[0].message.content.strip()}")

---
## 🌲 Homework / Next Step: Setting Up Pinecone
When you are ready to transition from local memory to managed cloud vector storage:

1. **Create an account:** Go to [https://app.pinecone.io/](https://app.pinecone.io/) and sign up for free (Starter Plan).
2. **Get your API Key:** In the Pinecone console, go to **API Keys** on the left menu and copy your key.
3. **Save in Google Colab:**
   * Click the 🔑 **Secrets** icon in the Colab left sidebar.
   * Add a new secret named `PINECONE_API_KEY` and paste your key.
   * Toggle **Notebook access** ON.
4. **Re-run the Notebook:** 
   * When you re-run Cell 1 and Cell 4, the notebook will detect your key, create a live `ServerlessSpec(cloud="aws", region="us-east-1")` index, and upsert your vectors to Pinecone Cloud!
   * You can open the Pinecone Web Console and actually see your 8 vectors and metadata stored in real time.